In [1]:
import soccerdata as sd
import pandas as pd

print("Libraries imported successfully")

# Test pull - Big 5, 2024-25 season
fbref = sd.FBref("Big 5 European Leagues Combined", ["2425"])

# Pull standard stats
df = fbref.read_player_season_stats(stat_type="standard")

print(f"Shape: {df.shape}")
print(df.head())

[07/01/26 21:46:55] INFO     No custom team name replacements found. You can configure these in       _config.py:92
                             /Users/ashish/soccerdata/config/teamname_replacements.json.                           

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:190
                             /Users/ashish/soccerdata/config/league_dict.json.                                     

Libraries imported successfully


                    INFO     Saving cached data to /Users/ashish/soccerdata/data/FBref               _common.py:250



*** chromedriver to download = 149.0.7827.155 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/149.0.7827.155/mac-x64/chromedriver-mac-x64.zip ...
Download Complete!

Extracting ['chromedriver'] from chromedriver-mac-x64.zip ...
Unzip Complete!

The file [uc_driver] was saved to:
/Users/ashish/Desktop/UoBD MSc Project/Football-PlayerValuation/venv/lib/python3.14/site-packages/seleniumbase/drivers/
uc_driver

Making [uc_driver 149.0.7827.155] executable ...
[uc_driver 149.0.7827.155] is now ready for use!


*** chromedriver to download = 149.0.7827.155 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/149.0.7827.155/mac-arm64/chromedriver-mac-arm64.zip ...
Download Complete!

Extracting ['chromedriver'] from chromedriver-mac-arm64.zip ...
Unzip Complete!

The file [chromedriver] was saved to:
/Users/ashish/Desktop/UoBD MSc Project/Football-PlayerValuation/venv/lib/python3.14/site-packages/seleniumbase/drivers/
chromedriver

Making

In [2]:
print(f"Total players: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"\nLeagues covered:")
print(df.index.get_level_values('league').unique())
print(f"\nColumns available:")
print(df.columns.tolist())

Total players: 2854
Total columns: 21

Leagues covered:
Index(['ENG-Premier League', 'ESP-La Liga', 'FRA-Ligue 1', 'ITA-Serie A', nan], dtype='str', name='league')

Columns available:
[('nation', ''), ('pos', ''), ('age', ''), ('born', ''), ('Playing Time', 'MP'), ('Playing Time', 'Starts'), ('Playing Time', 'Min'), ('Playing Time', '90s'), ('Performance', 'Gls'), ('Performance', 'Ast'), ('Performance', 'G+A'), ('Performance', 'G-PK'), ('Performance', 'PK'), ('Performance', 'PKatt'), ('Performance', 'CrdY'), ('Performance', 'CrdR'), ('Per 90 Minutes', 'Gls'), ('Per 90 Minutes', 'Ast'), ('Per 90 Minutes', 'G+A'), ('Per 90 Minutes', 'G-PK'), ('Per 90 Minutes', 'G+A-PK')]


In [4]:
stat_types = ["standard", "shooting", "keeper", "playing_time", "misc"]

all_stats = {}
for stat in stat_types:
    print(f"Pulling {stat} stats...")
    all_stats[stat] = fbref.read_player_season_stats(stat_type=stat)
    print(f"  Done — shape: {all_stats[stat].shape}")

print("\nAll stat types pulled successfully!")

Pulling standard stats...
  Done — shape: (2854, 21)
Pulling shooting stats...
  Done — shape: (2854, 15)
Pulling keeper stats...
  Done — shape: (212, 23)
Pulling playing_time stats...
  Done — shape: (3508, 21)
Pulling misc stats...
  Done — shape: (2854, 17)

All stat types pulled successfully!


In [5]:
import os

save_path = "../data/raw/fbref/"

for stat_name, df_stat in all_stats.items():
    filename = f"fbref_{stat_name}_2425.csv"
    df_stat.to_csv(os.path.join(save_path, filename))
    print(f"Saved {filename}")

print("\nAll FBref files saved!")

Saved fbref_standard_2425.csv
Saved fbref_shooting_2425.csv
Saved fbref_keeper_2425.csv
Saved fbref_playing_time_2425.csv
Saved fbref_misc_2425.csv

All FBref files saved!


In [7]:
import soccerdata as sd

fbref = sd.FBref("Big 5 European Leagues Combined", ["2425"])

# Check all available stat types
print(fbref.read_player_season_stats.__doc__)

[07/01/26 22:33:50] INFO     Saving cached data to /Users/ashish/soccerdata/data/FBref               _common.py:250

Retrieve players from the datasource for the selected leagues and seasons.

The following stat types are available:
    * 'standard'
    * 'shooting'
    * 'playing_time'
    * 'keeper'
    * 'misc'

Parameters
----------
stat_type :str
    Type of stats to retrieve.

Raises
------
TypeError
    If ``stat_type`` is not valid.

Returns
-------
pd.DataFrame



In [8]:
methods = [m for m in dir(fbref) if not m.startswith('_')]
print(methods)

['available_leagues', 'data_dir', 'get', 'headers', 'headless', 'leagues', 'max_delay', 'no_cache', 'no_store', 'path_to_browser', 'proxy', 'rate_limit', 'read_events', 'read_leagues', 'read_lineup', 'read_player_match_stats', 'read_player_season_stats', 'read_schedule', 'read_seasons', 'read_team_match_stats', 'read_team_season_stats', 'seasons']


In [9]:
import requests
import pandas as pd
import time

def scrape_fbref_stats(stat_type, season="2024-2025"):
    """
    Scrape player stats directly from FBref for Big 5 leagues.
    stat_type options: 'passing', 'defense', 'possession', 'gca', 'passing_types'
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
    }
    
    if season == "2024-2025":
        url = f"https://fbref.com/en/comps/Big5/{stat_type}/players/Big-5-European-Leagues-Stats"
    else:
        url = f"https://fbref.com/en/comps/Big5/{season}/{stat_type}/players/{season}-Big-5-European-Leagues-Stats"
    
    print(f"Scraping: {url}")
    
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed with status: {response.status_code}")
        return None
    
    # Read HTML tables
    tables = pd.read_html(response.text)
    
    # FBref player stats table is always the largest one
    df = max(tables, key=len)
    
    # Add season info
    df['season'] = season
    
    print(f"  Shape: {df.shape}")
    return df

# Test with passing stats for 2024-25
df_passing = scrape_fbref_stats("passing", "2024-2025")
print(df_passing.head())

Scraping: https://fbref.com/en/comps/Big5/passing/players/Big-5-European-Leagues-Stats
Failed with status: 403


AttributeError: 'NoneType' object has no attribute 'head'

In [10]:
# Check what stat types are available for team stats
help(fbref.read_team_season_stats)

Help on method read_team_season_stats in module soccerdata.fbref:

read_team_season_stats(
    stat_type: str = 'standard',
    opponent_stats: bool = False
) -> pd.DataFrame method of soccerdata.fbref.FBref instance
    Retrieve aggregated season stats for all teams in the selected leagues and seasons.

    The following stat types are available:
        * 'standard'
        * 'keeper'
        * 'shooting'
        * 'playing_time'
        * 'misc'

    Parameters
    ----------
    stat_type: str
        Type of stats to retrieve.
    opponent_stats: bool
        If True, will retrieve opponent stats.

    Raises
    ------
    ValueError
        If ``stat_type`` is not valid.

    Returns
    -------
    pd.DataFrame



In [11]:
help(fbref.read_player_match_stats)

Help on method read_player_match_stats in module soccerdata.fbref:

read_player_match_stats(
    stat_type: str = 'summary',
    match_id: Optional[Union[str, list[str]]] = None,
    force_cache: bool = False
) -> pd.DataFrame method of soccerdata.fbref.FBref instance
    Retrieve the match stats for the selected leagues and seasons.

    The following stat types are available:
        * 'summary'
        * 'keepers'

    Parameters
    ----------
    stat_type : str
        Type of stats to retrieve.
    match_id : int or list of int, optional
        Retrieve the event stream for a specific game.
    force_cache : bool
        By default no cached data is used to scrape the list of available
        games for the current season. If True, will force the use of
        cached data anyway.

    Raises
    ------
    ValueError
        If no games with the given IDs were found for the selected seasons and leagues.
    TypeError
        If ``stat_type`` is not valid.

    Returns
    ----

In [12]:
# Try using soccerdata's internal browser to fetch FBref pages
help(fbref.get)

Help on method get in module soccerdata._common:

get(
    url: str,
    filepath: Optional[Path] = None,
    max_age: Optional[Union[int, timedelta]] = None,
    no_cache: bool = False,
    var: Optional[Union[str, Iterable[str]]] = None
) -> IO[bytes] method of soccerdata.fbref.FBref instance
    Load data from `url`.

    By default, the source of `url` is downloaded and saved to `filepath`.
    If `filepath` exists, the `url` is not visited and the cached data is
    returned.

    Parameters
    ----------
    url : str
        URL to download.
    filepath : Path, optional
        Path to save downloaded file. If None, downloaded data is not cached.
    max_age : int for age in days, or timedelta object
        The max. age of locally cached file before re-download.
    no_cache : bool
        If True, will not use cached data. Overrides the class property.
    var : str or list of str, optional
        Return a JavaScript variable instead of the page source.

    Raises
    ----

In [13]:
import pandas as pd
from io import BytesIO

# Use soccerdata's browser to fetch passing stats
url = "https://fbref.com/en/comps/Big5/passing/players/Big-5-European-Leagues-Stats"

print("Fetching passing stats via soccerdata browser...")
response = fbref.get(url)
content = response.read()

# Parse the HTML tables
tables = pd.read_html(BytesIO(content))
df_passing = max(tables, key=len)

print(f"Shape: {df_passing.shape}")
print(df_passing.columns.tolist())
print(df_passing.head())

Fetching passing stats via soccerdata browser...
Shape: (2952, 30)
[('Unnamed: 0_level_0', 'Rk'), ('Unnamed: 1_level_0', 'Player'), ('Unnamed: 2_level_0', 'Nation'), ('Unnamed: 3_level_0', 'Pos'), ('Unnamed: 4_level_0', 'Squad'), ('Unnamed: 5_level_0', 'Comp'), ('Unnamed: 6_level_0', 'Age'), ('Unnamed: 7_level_0', 'Born'), ('Unnamed: 8_level_0', '90s'), ('Total', 'Cmp'), ('Total', 'Att'), ('Total', 'Cmp%'), ('Total', 'TotDist'), ('Total', 'PrgDist'), ('Short', 'Cmp'), ('Short', 'Att'), ('Short', 'Cmp%'), ('Medium', 'Cmp'), ('Medium', 'Att'), ('Medium', 'Cmp%'), ('Long', 'Cmp'), ('Long', 'Att'), ('Long', 'Cmp%'), ('Unnamed: 23_level_0', 'Ast'), ('Unnamed: 24_level_0', 'A-xAG'), ('Unnamed: 25_level_0', 'KP'), ('Unnamed: 26_level_0', '1/3'), ('Unnamed: 27_level_0', 'PPA'), ('Unnamed: 28_level_0', 'CrsPA'), ('Unnamed: 29_level_0', 'Matches')]
  Unnamed: 0_level_0   Unnamed: 1_level_0 Unnamed: 2_level_0  \
                  Rk               Player             Nation   
0                  1 

In [14]:
import pandas as pd
from io import BytesIO
import time
import os

def scrape_fbref_big5(stat_type, season_url_part=""):
    """
    Scrape any FBref stat type for Big 5 leagues using soccerdata's browser.
    """
    if season_url_part == "":
        # Current/most recent season
        url = f"https://fbref.com/en/comps/Big5/{stat_type}/players/Big-5-European-Leagues-Stats"
    else:
        url = f"https://fbref.com/en/comps/Big5/{season_url_part}/{stat_type}/players/{season_url_part}-Big-5-European-Leagues-Stats"
    
    print(f"  Fetching: {url}")
    response = fbref.get(url)
    content = response.read()
    
    tables = pd.read_html(BytesIO(content))
    df = max(tables, key=len)
    
    # Flatten multi-level columns
    df.columns = ['_'.join(col).strip('_') for col in df.columns]
    
    return df

# Test all stat types for 2024-25
stat_types = {
    "passing": "",
    "defense": "",
    "possession": "",
    "gca": "",      # Goal creating actions
    "passing_types": ""
}

save_path = "../data/raw/fbref/"

for stat, season in stat_types.items():
    print(f"Pulling {stat}...")
    df = scrape_fbref_big5(stat, season)
    print(f"  Shape: {df.shape}")
    df.to_csv(os.path.join(save_path, f"fbref_{stat}_2425.csv"), index=False)
    print(f"  Saved!")
    time.sleep(4)  # Be respectful to FBref servers

print("\nAll additional stat types pulled and saved!")

Pulling passing...
  Fetching: https://fbref.com/en/comps/Big5/passing/players/Big-5-European-Leagues-Stats
  Shape: (2952, 30)
  Saved!
Pulling defense...
  Fetching: https://fbref.com/en/comps/Big5/defense/players/Big-5-European-Leagues-Stats
  Shape: (2952, 26)
  Saved!
Pulling possession...
  Fetching: https://fbref.com/en/comps/Big5/possession/players/Big-5-European-Leagues-Stats
  Shape: (2952, 30)
  Saved!
Pulling gca...
  Fetching: https://fbref.com/en/comps/Big5/gca/players/Big-5-European-Leagues-Stats
  Shape: (2952, 26)
  Saved!
Pulling passing_types...
  Fetching: https://fbref.com/en/comps/Big5/passing_types/players/Big-5-European-Leagues-Stats
  Shape: (2952, 25)
  Saved!

All additional stat types pulled and saved!
